# URI, URL, and URN - Python

All 9 Python examples from [docs/core/uri.md](https://platob.github.io/yggdryl/core/uri/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import Uri

uri = Uri("HTTPS://example.test/archive/report.tar.gz?q=1#summary")

assert str(uri) == "https://example.test/archive/report.tar.gz?q=1#summary"
assert uri.scheme == "https"
assert uri.authority == "example.test"
assert uri.path == "/archive/report.tar.gz"
assert uri.query == "q=1"
assert uri.fragment == "summary"
assert uri.file_name == "report.tar.gz"

## Canonical on arrival

In [ ]:
from yggdryl import Uri

uri = Uri("HTTPS://example.test/caf%c3%a9.csv")
assert str(uri) == "https://example.test/caf%C3%A9.csv"

windows = Uri(r"file:///C:\Users\Ada\report.parquet")
assert str(windows) == "file:///C:/Users/Ada/report.parquet"

assert str(Uri("/var/lib/data.arrow")) == "file:///var/lib/data.arrow"
assert str(Uri("data/ticks.csv")) == "file:data/ticks.csv"

stamped = Uri("/data/2026-08-16T00:00:00/part.parquet")
assert stamped.scheme == "file"

assert Uri(str(uri)) == uri

## Path segments

In [ ]:
from yggdryl import Uri

uri = Uri("https://example.test/archive/2026/report.tar.gz")

assert uri.path_segments == ("archive", "2026", "report.tar.gz")
assert len(uri) == 3
assert uri[0] == "archive"
assert uri[-1] == "report.tar.gz"
assert "2026" in uri
assert list(uri) == list(uri.path_segments)

## Compound filenames

In [ ]:
from yggdryl import Uri

uri = Uri("https://example.test/archive/report.tar.gz?q=1#part")

assert uri.file_name == "report.tar.gz"
assert uri.stem == "report.tar"
assert uri.extension == "gz"
assert uri.extensions == ("tar", "gz")

uri.set_stem("renamed")
assert str(uri) == "https://example.test/archive/renamed.gz?q=1#part"
uri.set_extensions(["csv", "gz"])
assert str(uri) == "https://example.test/archive/renamed.csv.gz?q=1#part"
assert uri.remove_extension() is True
assert uri.clear_extensions() is True
assert str(uri) == "https://example.test/archive/renamed?q=1#part"

unchanged = str(uri)
try:
    uri.set_file_name("bad/name")
    raise AssertionError("a separator is not a filename")
except ValueError:
    pass
assert str(uri) == unchanged

## The media type is in the name

In [ ]:
from yggdryl import MediaType, MimeType, Uri

uri = Uri("https://example.test/report.csv.gz.zst?q=1#part")

assert uri.mime_type == MimeType("application/zstd")
assert uri.media_type.base == MimeType("text/csv")
assert uri.media_type.encodings == (
    MimeType("application/gzip"),
    MimeType("application/zstd"),
)

uri.set_mime_type("application/json")
assert str(uri) == "https://example.test/report.csv.gz.json?q=1#part"

uri.set_media_type(
    MediaType.from_parts(
        MimeType("text/csv"),
        [MimeType("application/gzip"), MimeType("application/zstd")],
    )
)
assert str(uri) == "https://example.test/report.csv.gz.zst?q=1#part"

unchanged = str(uri)
try:
    uri.set_mime_type("application/vnd.example")
    raise AssertionError("no preferred filename extension")
except ValueError:
    pass
assert str(uri) == unchanged

## URL and URN

In [ ]:
from yggdryl import Uri, Url, Urn

uri = Uri("https://example.test/a/data.json?raw=true")
url = Url(uri)
assert url.authority == "example.test"
assert Uri(url) == uri

urn = Urn("URN:ISBN:9780131103627")
assert str(urn) == "urn:isbn:9780131103627"
assert urn.namespace == "isbn"
assert urn.namespace_specific == "9780131103627"
assert urn.authority == ""

for rejected in (
    lambda: urn.to_uri().to_url(),
    lambda: Urn(uri),
    lambda: Url("mailto:user@example.test"),
):
    try:
        rejected()
        raise AssertionError("expected a rejection")
    except ValueError:
        pass

## Platform paths

In [ ]:
import os
from pathlib import PureWindowsPath

from yggdryl import Uri, Url

uri = Uri.from_path(PureWindowsPath(r"C:\Users\Ada Lovelace\report.parquet"))
assert str(uri) == "file:///C:/Users/Ada%20Lovelace/report.parquet"
assert uri.authority == ""
assert uri.file_name == "report.parquet"

# `__fspath__` is `to_path`, so a URI goes straight into `open` or `pathlib`.
assert uri.to_path() == "C:/Users/Ada Lovelace/report.parquet"
assert os.fspath(uri) == uri.to_path()
assert Uri.from_path(uri.to_path()) == uri

unc = Uri.from_path(r"\\server\share\prices\ticks.csv")
assert str(unc) == "file://server/share/prices/ticks.csv"
assert unc.authority == "server"
assert unc.to_path() == "//server/share/prices/ticks.csv"

try:
    Url("https://example.test/data.csv").to_path()
    raise AssertionError("a network URL has no path")
except ValueError:
    pass

## Walking the path

In [ ]:
from yggdryl import Url

url = Url("https://example.test/a/b/c?q=1#frag")

# `joinpath` and `/` compose the way they do on a `PurePath`.
assert str(url / "d") == "https://example.test/a/b/c/d?q=1#frag"
assert str(url.joinpath("d", "e")) == "https://example.test/a/b/c/d/e?q=1#frag"

# `parts` is the sequence of names the path actually addresses.
assert url.parts == ("a", "b", "c")

# `parents` climbs to the root and never yields the value itself.
assert [parent.path for parent in url.parents] == ["/a/b", "/a", "/"]
assert url.parent.path == "/a/b"

## Patterns and partitions

In [ ]:
from yggdryl import Url

pattern = Url("file:///lake/trades/year=2024/**/*.parquet")
assert pattern.is_glob()

part = Url("file:///lake/trades/year=2024/month=01/part-0.parquet")
assert part.match("*.parquet")
assert part.match("lake/**/part-?.parquet")
assert not part.match("lake/*.parquet")

assert part.partition("month") == "01"
assert part.partitions == (("year", "2024"), ("month", "01"))
assert part.relative_to(Url("file:///lake/trades")) == "year=2024/month=01/part-0.parquet"